# Bias and fairness

Exercise notebook for the lesson **[Bias and fairness](https://ramkumarmd.com/medical-ai/evaluation-safety/bias-and-fairness)** in the *Evaluation, safety & deployment* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** Clinical AI can treat groups of patients unequally even when no one intends it: through labels that measure access to care rather than health need, through data that under-represent some groups, or through models that are well calibrated overall but not within each group. Leaving group membership out of a model does not make it fair. Measure discrimination, calibration, sensitivity and predictive values in each relevant subgroup, choose fairness goals deliberately, and document a dataset's limits, as the STANDING Together recommendations advise.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Label bias, in code

Equal need, unequal access: who gets selected when spending is the label?

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 20000
group = rng.choice(["A", "B"], n, p=[0.7, 0.3])
need = rng.gamma(2.0, 1.0, n)                   # true health need (same distribution in both groups)
access = np.where(group == "A", 1.0, 0.6)       # group B spends less for the same need
spending = need * access * rng.lognormal(0, 0.3, n)
df = pd.DataFrame({"group": group, "need": need, "spending": spending})

selected = df.spending >= df.spending.quantile(0.90)
high_need = df.need >= df.need.quantile(0.90)
summary = pd.DataFrame({
    "share of population": df.group.value_counts(normalize=True),
    "share of high-need patients": df[high_need].group.value_counts(normalize=True),
    "share of patients selected": df[selected].group.value_counts(normalize=True),
}).round(2)
print(summary.to_string())
print(f"\nHigh-need patients missed: group A {1 - selected[high_need & (df.group == 'A')].mean():.0%}, "
      f"group B {1 - selected[high_need & (df.group == 'B')].mean():.0%}")

#        share of population  share of high-need patients  share of patients selected
# group
# A                      0.7                         0.68                        0.91
# B                      0.3                         0.32                        0.09
#
# High-need patients missed: group A 24%, group B 75%

### Subgroup performance, in code

A group-blind model: calibration, sensitivity and PPV by group at a 20% threshold.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(8)
n = 20000
group = rng.choice(["A", "B"], n, p=[0.8, 0.2])
x = rng.normal(size=(n, 3))
baseline = np.where(group == "A", -2.0, -1.0)          # outcome more common in group B
y = rng.binomial(1, 1 / (1 + np.exp(-(baseline + x @ np.array([1.0, 0.7, 0.4])))))

def subgroup_table(p, threshold=0.2):
    flag = p >= threshold
    rows = []
    for g in ["A", "B"]:
        m = group == g
        tp, fn = np.sum(flag & (y == 1) & m), np.sum(~flag & (y == 1) & m)
        fp = np.sum(flag & (y == 0) & m)
        rows.append({"group": g, "outcome rate": y[m].mean(), "mean predicted": p[m].mean(),
                     "sensitivity": tp / (tp + fn), "PPV": tp / (tp + fp)})
    return pd.DataFrame(rows).round(2).to_string(index=False)

p = LogisticRegression().fit(x, y).predict_proba(x)[:, 1]   # group not used
print(subgroup_table(p))

# group  outcome rate  mean predicted  sensitivity  PPV
#     A          0.18             0.2         0.75 0.34
#     B          0.31             0.2         0.67 0.55

### Fairness criteria can conflict

Continuing from the previous block: the same model allowed to use group membership.

In [ ]:
X_with_group = np.column_stack([x, group == "B"])
p2 = LogisticRegression().fit(X_with_group, y).predict_proba(X_with_group)[:, 1]
print(subgroup_table(p2))

# group  outcome rate  mean predicted  sensitivity  PPV
#     A          0.18            0.18         0.67 0.37
#     B          0.31            0.31         0.87 0.45

## Exercise

**Goal:** Audit a synthetic model for label bias and subgroup performance, and explore fairness trade-offs.

**Data:** Synthetic populations generated in the notebook (no real patient data) (Generated in the notebook)

### Step 1

Reproduce the label-bias simulation, then select by need instead of spending and compare the selected shares.

In [ ]:
# Your code here


### Step 2

Reproduce the subgroup tables and add AUROC and calibration plots per group.

In [ ]:
# Your code here


### Step 3

Choose group-specific thresholds that equalise sensitivity, and report what happens to PPV and false-positive rates.

In [ ]:
# Your code here


### Step 4

Shrink group B to 5% of the data and see how uncertain its metrics become (bootstrap confidence intervals).

In [ ]:
# Your code here


### Step 5

Write a one-paragraph fairness statement for this model, saying which goal you prioritised and why.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/evaluation-safety/bias-and-fairness](https://ramkumarmd.com/medical-ai/evaluation-safety/bias-and-fairness). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.